# EasyVisa: 2026 re-run with a sound evaluation protocol

**Author:** Mersedyes Livingston · re-run November 2026 · original project completed April 2025

This notebook re-does the 2025 visa-certification project (the original executed notebook was not recovered). It keeps the 2025 preprocessing choices and fixes the evaluation problems found in the [2026 re-audit](../README.md):

| 2025 | 2026 re-run |
|---|---|
| Headline metric: F1 on the *certified* class | Positive-class F1 **plus** the trivial "approve everyone" baseline, balanced accuracy, macro-F1, and ROC-AUC |
| Over-/under-sampling, plus `scale_pos_weight` on already-balanced data | Original class balance; the decision threshold is set explicitly instead |
| Final model reported without uncertainty | Bootstrap 95% confidence intervals on the test set |
| "Improves fairness" claimed, never measured | Error rates broken down by applicant continent, plus a model retrained **without** continent |

**Protocol (fixed before running):**
- Stratified 70/30 split.
- All tuning and model selection by 5-fold stratified cross-validation on the training set.
- Selection metric: ROC-AUC.
- Test set used once for the selected model; the continent ablation reuses it as a sensitivity analysis, not for selection.

**Data:** not redistributed. Put `EasyVisa.csv` in `data/` at the repository root.


In [1]:
import hashlib, json, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, make_scorer,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="A worker stopped")  # joblib worker restarts: harmless
SEED = 1
DATA = Path("../../data/EasyVisa.csv")
OUT_DIR = Path(".")
pd.set_option("display.float_format", "{:.3f}".format)
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__}")

Python 3.13.18 | scikit-learn 1.9.1 | xgboost 3.4.1 | pandas 3.0.5


## 1. Load and validate

In [2]:
REQUIRED = ["case_id", "continent", "education_of_employee", "has_job_experience", "requires_job_training",
            "no_of_employees", "yr_of_estab", "region_of_employment", "prevailing_wage", "unit_of_wage",
            "full_time_position", "case_status"]
if not DATA.is_file():
    raise FileNotFoundError(f"Put EasyVisa.csv at {DATA.resolve()} (data is not redistributed)")
raw = pd.read_csv(DATA)
missing = [c for c in REQUIRED if c not in raw.columns]
assert not missing, f"missing columns: {missing}"
assert raw[REQUIRED].notna().all().all(), "unexpected missing values"
assert set(raw.case_status) == {"Certified", "Denied"}
DATA_SHA256 = hashlib.sha256(DATA.read_bytes()).hexdigest()

print(f"{len(raw):,} cases | certified {(raw.case_status == 'Certified').mean():.1%} | "
      f"negative employee counts: {(raw.no_of_employees < 0).sum()} | duplicates: {raw.duplicated().sum()}")
print("data sha256:", DATA_SHA256)

25,480 cases | certified 66.8% | negative employee counts: 33 | duplicates: 0
data sha256: 84eff9003d858ca52386f233f6e285ebd87486ef36580ea77a44826166308c0c


## 2. Preprocessing (same choices as 2025)
- **Drop `case_id`:** it is an identifier.
- **Negative employee counts:** treated as sign errors (absolute value).
- **Annualize wages:** hour × 2,080; week × 52; month × 12. The unit itself is kept as a feature.
- **Target:** certified = 1.

In [3]:
ANNUAL = {"Hour": 2080, "Week": 52, "Month": 12, "Year": 1}
CATEGORICAL = ["continent", "education_of_employee", "has_job_experience", "requires_job_training",
               "region_of_employment", "unit_of_wage", "full_time_position"]
NUMERIC = ["no_of_employees", "yr_of_estab", "annualized_wage"]

def prepare(df):
    df = df.copy()
    df["no_of_employees"] = df["no_of_employees"].abs()
    df["annualized_wage"] = df["prevailing_wage"] * df["unit_of_wage"].map(ANNUAL)
    assert df["annualized_wage"].notna().all(), "unknown unit_of_wage value"
    return df[CATEGORICAL + NUMERIC], (df["case_status"] == "Certified").astype(int)

X, y = prepare(raw)
X.describe().T

,count,mean,std,min,25%,50%,75%,max
no_of_employees,25480.000,5667.089,22877.917,11.000,1022.000,2109.000,3504.000,602069.000
yr_of_estab,25480.000,1979.410,42.367,1800.000,1976.000,1997.000,2005.000,2016.000
annualized_wage,25480.000,197391.155,578591.666,100.000,47107.965,82839.460,124825.035,14569149.400


## 3. Split and cross-validation setup

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def preprocessor(cats=CATEGORICAL, nums=NUMERIC):
    return ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cats),
                              ("num", StandardScaler(), nums)])

SCORING = {"roc_auc": "roc_auc", "balanced_accuracy": "balanced_accuracy",
           "macro_f1": make_scorer(f1_score, average="macro"),
           "f1_certified": make_scorer(f1_score, zero_division=0),
           "recall_denied": make_scorer(recall_score, pos_label=0, zero_division=0),
           "accuracy": "accuracy"}
print(f"train {len(X_tr):,} | test {len(X_te):,} | certified rate train {y_tr.mean():.3f} test {y_te.mean():.3f}")

train 17,836 | test 7,644 | certified rate train 0.668 test 0.668


## 4. Candidates, tuned by cross-validation on the training set only
- The baseline approves everyone: the rule positive-class F1 cannot distinguish from a model.
- Tree ensembles mirror the 2025 shortlist (bagging/random forest, AdaBoost, gradient boosting, XGBoost). Logistic regression is added as a transparent reference.

In [5]:
CANDIDATES = {
    "Baseline: approve everyone": (DummyClassifier(strategy="most_frequent"), {}),
    "Logistic regression": (LogisticRegression(max_iter=5000), {"clf__C": [0.1, 1, 10], "clf__class_weight": [None, "balanced"]}),
    "Random forest": (RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
                      {"clf__max_depth": [None, 12], "clf__min_samples_leaf": [1, 5]}),
    "AdaBoost": (AdaBoostClassifier(random_state=SEED), {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.1, 1.0]}),
    "Gradient boosting": (GradientBoostingClassifier(n_estimators=150, random_state=SEED),
                          {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=SEED, n_jobs=-1),
                {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]}),
}

fitted, rows = {}, []
for name, (clf, grid) in CANDIDATES.items():
    pipe = Pipeline([("prep", preprocessor()), ("clf", clf)])
    if grid:
        search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
        best, params = search.best_estimator_, {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    else:
        best, params = pipe.fit(X_tr, y_tr), {}
    s = cross_validate(best, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=-1)
    fitted[name] = best
    rows.append({"model": name, **{m: s[f"test_{m}"].mean() for m in SCORING},
                 "roc_auc_sd": s["test_roc_auc"].std(), "params": params})

cv_table = pd.DataFrame(rows).set_index("model").sort_values("roc_auc", ascending=False)
cv_table

,roc_auc,balanced_accuracy,macro_f1,f1_certified,recall_denied,accuracy,roc_auc_sd,params
model,,,,,,,,
Random forest,0.787,0.686,0.696,0.824,0.495,0.750,0.004,"{'max_depth': 12, 'min_samples_leaf': 5}"
Gradient boosting,0.783,0.685,0.695,0.823,0.497,0.749,0.005,"{'learning_rate': 0.05, 'max_depth': 3}"
XGBoost,0.782,0.686,0.696,0.823,0.498,0.749,0.005,"{'learning_rate': 0.05, 'max_depth': 3}"
Logistic regression,0.772,0.695,0.679,0.755,0.690,0.697,0.005,"{'C': 1, 'class_weight': 'balanced'}"
AdaBoost,0.771,0.663,0.673,0.818,0.442,0.737,0.005,"{'learning_rate': 1.0, 'n_estimators': 100}"
Baseline: approve everyone,0.500,0.500,0.400,0.801,0.000,0.668,0.000,{}


## 5. Select, set the threshold on training folds, then test once
- The model with the best cross-validated ROC-AUC is selected.
- The threshold that maximizes balanced accuracy is chosen on **out-of-fold training predictions**. Resampling in 2025 was an indirect way of moving this threshold; here it is explicit.

In [6]:
final_name = cv_table.drop(index="Baseline: approve everyone")["roc_auc"].idxmax()
final = fitted[final_name]
oof = cross_val_predict(final, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
fpr, tpr, thr = roc_curve(y_tr, oof)
THRESHOLD = float(thr[np.argmax(tpr - fpr)])  # maximizes balanced accuracy (Youden's J)
print(f"selected: {final_name} | threshold from training folds: {THRESHOLD:.3f}")

final.fit(X_tr, y_tr)
p_te = final.predict_proba(X_te)[:, 1]

def metrics(y_true, pred, score=None):
    out = {"accuracy": accuracy_score(y_true, pred), "balanced_accuracy": balanced_accuracy_score(y_true, pred),
           "macro_f1": f1_score(y_true, pred, average="macro"), "f1_certified": f1_score(y_true, pred, zero_division=0),
           "recall_certified": recall_score(y_true, pred, zero_division=0),
           "recall_denied": recall_score(y_true, pred, pos_label=0, zero_division=0),
           "precision_certified": precision_score(y_true, pred, zero_division=0)}
    if score is not None:
        out["roc_auc"] = roc_auc_score(y_true, score)
    return out

def bootstrap(y_true, pred, score=None, n=2000, seed=SEED):
    rng, y_true, pred = np.random.default_rng(seed), np.asarray(y_true), np.asarray(pred)
    score = None if score is None else np.asarray(score)
    draws = [metrics(y_true[i], pred[i], None if score is None else score[i])
             for i in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n))]
    return {k: (np.percentile([d[k] for d in draws], 2.5), np.percentile([d[k] for d in draws], 97.5)) for k in draws[0]}

approve_all = np.ones_like(y_te)
test = {"Baseline: approve everyone": metrics(y_te, approve_all),
        f"{final_name} @ 0.5": metrics(y_te, (p_te >= 0.5).astype(int), p_te),
        f"{final_name} @ {THRESHOLD:.3f}": metrics(y_te, (p_te >= THRESHOLD).astype(int), p_te)}
ci = {"@0.5": bootstrap(y_te, (p_te >= 0.5).astype(int), p_te), "@thr": bootstrap(y_te, (p_te >= THRESHOLD).astype(int), p_te)}
test_table = pd.DataFrame(test).T
test_table

selected: Random forest | threshold from training folds: 0.646


,accuracy,balanced_accuracy,macro_f1,f1_certified,recall_certified,recall_denied,precision_certified,roc_auc
Baseline: approve everyone,0.668,0.500,0.400,0.801,1.000,0.000,0.668,NaN
Random forest @ 0.5,0.745,0.678,0.689,0.821,0.878,0.479,0.772,0.777
Random forest @ 0.646,0.724,0.700,0.696,0.789,0.771,0.629,0.807,0.777


In [7]:
ci_table = pd.DataFrame({k: {m: f"[{lo:.3f}, {hi:.3f}]" for m, (lo, hi) in v.items()} for k, v in ci.items()}).T
ci_table.index = [f"{final_name} @ 0.5 (95% CI)", f"{final_name} @ {THRESHOLD:.3f} (95% CI)"]
ci_table

,accuracy,balanced_accuracy,macro_f1,f1_certified,recall_certified,recall_denied,precision_certified,roc_auc
Random forest @ 0.5 (95% CI),"[0.735, 0.755]","[0.668, 0.689]","[0.677, 0.700]","[0.814, 0.829]","[0.869, 0.886]","[0.460, 0.498]","[0.761, 0.783]","[0.767, 0.788]"
Random forest @ 0.646 (95% CI),"[0.714, 0.734]","[0.689, 0.711]","[0.685, 0.706]","[0.780, 0.797]","[0.760, 0.783]","[0.610, 0.647]","[0.796, 0.818]","[0.767, 0.788]"


## 6. Why positive-class F1 misled the 2025 evaluation
With 66.8% of cases certified, a rule that approves everyone gets recall 1.0 and precision equal to the base rate. Its positive-class F1 is therefore high even though it has no skill. Balanced accuracy and macro-F1 give that rule 0.5 and about 0.40, which exposes it.

In [8]:
b, m05 = test["Baseline: approve everyone"], test[f"{final_name} @ 0.5"]
for k in ["f1_certified", "balanced_accuracy", "macro_f1", "accuracy"]:
    print(f"{k:18s} baseline {b[k]:.3f} | {final_name} @0.5 {m05[k]:.3f} | difference {m05[k] - b[k]:+.3f}")

f1_certified       baseline 0.801 | Random forest @0.5 0.821 | difference +0.021
balanced_accuracy  baseline 0.500 | Random forest @0.5 0.678 | difference +0.178
macro_f1           baseline 0.400 | Random forest @0.5 0.689 | difference +0.288
accuracy           baseline 0.668 | Random forest @0.5 0.745 | difference +0.077


## 7. Error rates by applicant continent
This is a descriptive disparity check, not a full fairness audit.
- **Selected-model approval rate:** how often the model predicts certification.
- **False approval rate:** denied cases predicted certified.
- **False denial rate:** certified cases predicted denied.

Small groups (Oceania, Africa) have wide uncertainty, so read their rates alongside `n`.

In [9]:
def by_group(groups, y_true, pred):
    df = pd.DataFrame({"g": groups.values, "y": np.asarray(y_true), "p": np.asarray(pred)})
    out = df.groupby("g").apply(lambda d: pd.Series({
        "n": len(d), "actual_certified_rate": d.y.mean(), "predicted_certified_rate": d.p.mean(),
        "false_approval_rate": ((d.p == 1) & (d.y == 0)).sum() / max((d.y == 0).sum(), 1),
        "false_denial_rate": ((d.p == 0) & (d.y == 1)).sum() / max((d.y == 1).sum(), 1),
        "accuracy": (d.p == d.y).mean()}), include_groups=False)
    out["n"] = out["n"].astype(int)
    return out.sort_values("n", ascending=False)

pred_thr = (p_te >= THRESHOLD).astype(int)
group_with = by_group(X_te["continent"], y_te, pred_thr)
group_with

,n,actual_certified_rate,predicted_certified_rate,false_approval_rate,false_denial_rate,accuracy
g,,,,,,
Asia,5061,0.651,0.631,0.378,0.234,0.716
Europe,1104,0.792,0.841,0.483,0.064,0.849
North America,1021,0.634,0.513,0.318,0.374,0.646
South America,237,0.582,0.414,0.202,0.435,0.662
Africa,154,0.708,0.630,0.356,0.257,0.714
Oceania,67,0.627,0.582,0.320,0.262,0.716


## 8. Ablation: the same model retrained without `continent`
- The selected configuration is refit **without** the continent feature, at the same threshold rule (set on its own training folds).
- If overall performance barely changes, continent is not needed for accuracy and its use needs separate justification.
- Removing a feature does not remove its proxies (e.g., wage or education patterns that differ by origin). That is why the per-group rates are reported again.

In [10]:
cats_wo = [c for c in CATEGORICAL if c != "continent"]
final_wo = Pipeline([("prep", preprocessor(cats_wo)), ("clf", final.named_steps["clf"].__class__(**final.named_steps["clf"].get_params()))])
X_tr_wo, X_te_wo = X_tr.drop(columns="continent"), X_te.drop(columns="continent")
cv_wo = cross_validate(final_wo, X_tr_wo, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
oof_wo = cross_val_predict(final_wo, X_tr_wo, y_tr, cv=cv, method="predict_proba")[:, 1]
f_, t_, th_ = roc_curve(y_tr, oof_wo); THRESHOLD_WO = float(th_[np.argmax(t_ - f_)])
final_wo.fit(X_tr_wo, y_tr)
p_te_wo = final_wo.predict_proba(X_te_wo)[:, 1]
pred_wo = (p_te_wo >= THRESHOLD_WO).astype(int)
ablation = pd.DataFrame({"with continent": metrics(y_te, pred_thr, p_te),
                         "without continent": metrics(y_te, pred_wo, p_te_wo)}).T
print(f"CV ROC-AUC without continent: {cv_wo.mean():.3f} ± {cv_wo.std():.3f} (with: {cv_table.loc[final_name, 'roc_auc']:.3f})")
ablation

CV ROC-AUC without continent: 0.768 ± 0.005 (with: 0.787)


,accuracy,balanced_accuracy,macro_f1,f1_certified,recall_certified,recall_denied,precision_certified,roc_auc
with continent,0.724,0.700,0.696,0.789,0.771,0.629,0.807,0.777
without continent,0.712,0.685,0.681,0.780,0.765,0.605,0.795,0.760


In [11]:
group_wo = by_group(X_te["continent"], y_te, pred_wo)
compare = pd.concat({"with continent": group_with[["n", "actual_certified_rate", "predicted_certified_rate", "false_denial_rate"]],
                     "without continent": group_wo[["predicted_certified_rate", "false_denial_rate"]]}, axis=1)
compare

with continent                                                 \
                           n actual_certified_rate predicted_certified_rate   
g                                                                             
Asia                    5061                 0.651                    0.631   
Europe                  1104                 0.792                    0.841   
North America           1021                 0.634                    0.513   
South America            237                 0.582                    0.414   
Africa                   154                 0.708                    0.630   
Oceania                   67                 0.627                    0.582   

                                       without continent                    
              false_denial_rate predicted_certified_rate false_denial_rate  
g                                                                           
Asia                      0.234                    0.644             0.225  
Europe                    0.064                    0.668             0.247  
North America             0.374                    0.631             0.261  
South America             0.435                    0.553             0.283  
Africa                    0.257                    0.649             0.220  
Oceania                   0.262                    0.582             0.262

## 9. What drives the selected model (permutation importance on the test set, ROC-AUC drop)

In [12]:
pi = permutation_importance(final, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
importance = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
importance.to_frame("mean ROC-AUC drop")

,mean ROC-AUC drop
education_of_employee,0.147
has_job_experience,0.064
continent,0.031
region_of_employment,0.027
unit_of_wage,0.022
annualized_wage,0.004
full_time_position,0.003
requires_job_training,0.002
no_of_employees,0.001
yr_of_estab,-0.001


## 10. Save results with provenance

In [13]:
results = {
    "data_sha256": DATA_SHA256, "n_rows": int(len(X)), "certified_rate": float(y.mean()),
    "selection_metric": "ROC-AUC, 5-fold stratified CV on training set",
    "cv": cv_table.reset_index().to_dict(orient="records"),
    "final_model": final_name, "threshold": THRESHOLD,
    "test": {k: v for k, v in test.items()},
    "test_ci95": {k: {m: list(map(float, v)) for m, v in d.items()} for k, d in ci.items()},
    "by_continent_with": group_with.reset_index().to_dict(orient="records"),
    "ablation_without_continent": {"cv_roc_auc_mean": float(cv_wo.mean()), "threshold": THRESHOLD_WO,
                                   "test": ablation.loc["without continent"].to_dict(),
                                   "by_continent": group_wo.reset_index().to_dict(orient="records")},
    "permutation_importance": importance.to_dict(),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "xgboost": xgboost.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
(OUT_DIR / "results.json").write_text(json.dumps(results, indent=2, default=float) + "\n")
print("wrote results.json")

wrote results.json
